# Task 2B — QLoRA fine-tuning (Colab T4)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krishanSKDA/CDAZZDEV-MLE-Krishan_Danushka/blob/main/task2_genai/Task2_FineTune.ipynb)

Runtime → Change runtime type → **T4 GPU**. Needs `HF_TOKEN` (write) in Colab Secrets; `WANDB_API_KEY` optional.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/krishanSKDA/CDAZZDEV-MLE-Krishan_Danushka.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/repo"], check=True)
    os.chdir("/content/repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "task2_genai/requirements-train.txt"], check=True)
else:
    while not os.path.exists("requirements.txt"):
        os.chdir("..")
sys.path.insert(0, os.getcwd())
print("repo root:", os.getcwd(), "| colab:", IN_COLAB)

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from common.config import get_secret
HF_TOKEN = get_secret("HF_TOKEN")
WANDB_KEY = get_secret("WANDB_API_KEY", required=False)
from huggingface_hub import HfApi
HF_REPO_ID = f"{HfApi(token=HF_TOKEN).whoami()['name']}/qwen2.5-3b-risk-extractor"
print("will push to:", HF_REPO_ID)
REPORT_TO = "wandb" if WANDB_KEY else "none"
if WANDB_KEY:
    import wandb; wandb.login(key=WANDB_KEY); os.environ["WANDB_PROJECT"] = "cdazzdev-task2"
print("report_to:", REPORT_TO)

In [ ]:
from task2_genai.dataset import DATA_DIR, load_jsonl
from task2_genai.finetune import TrainConfig, load_tokenizer, token_length_stats

train, val = load_jsonl(DATA_DIR / "train.jsonl"), load_jsonl(DATA_DIR / "val.jsonl")
cfg = TrainConfig()
tokenizer = load_tokenizer(cfg)
stats = token_length_stats(tokenizer, train + val)
cfg.max_seq_length = stats["suggested_max_seq_length"]
print("examples:", len(train), "train /", len(val), "val")
print("token lengths:", stats)

## Hyperparameters — every value justified

In [ ]:
import pandas as pd
from task2_genai.finetune import hyperparameter_table
pd.set_option("display.max_colwidth", None)
hyperparameter_table(cfg)

## Load the 4-bit NF4 base model and attach LoRA adapters

In [ ]:
from task2_genai.finetune import load_qlora_model
model = load_qlora_model(cfg)
model.print_trainable_parameters()
print(model.base_model.model.config.quantization_config)
print(f"GPU memory after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Train (validation every epoch, best checkpoint kept)

In [ ]:
from task2_genai.finetune import build_trainer
OUT = "/content/outputs" if IN_COLAB else "outputs_model"
trainer = build_trainer(cfg, model, tokenizer, train, val, f"{OUT}/checkpoints", REPORT_TO)
train_result = trainer.train()
print(train_result.metrics)
print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

## Loss per epoch

In [ ]:
import json, matplotlib.pyplot as plt
from task2_genai.finetune import epoch_loss_table
table = epoch_loss_table(trainer.state.log_history)
print(table.to_string(index=False))
print("validation loss strictly decreasing:", bool((table["val_loss"].diff().dropna() < 0).all()))
steps = pd.DataFrame([h for h in trainer.state.log_history if "loss" in h and "eval_loss" not in h])
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(steps["epoch"], steps["loss"], label="train loss (per log step)", alpha=0.6)
ax.plot(table["epoch"], table["val_loss"], "o-", label="val loss (per epoch)")
ax.set_xlabel("epoch"); ax.legend(); plt.show()
os.makedirs("task2_genai/results", exist_ok=True)
table.to_csv("task2_genai/results/epoch_losses.csv", index=False)
json.dump(trainer.state.log_history, open("task2_genai/results/log_history.json", "w"), indent=1)

## Out-of-memory log
Document any OOM here: the exact error, what was tried, and the fix applied. Preventive measures already in place: 4-bit NF4 + double quantisation, gradient checkpointing, paged 8-bit AdamW, batch 2 with accumulation 4, data-driven max length, and freeing the training model before the fp16 merge.

## Save adapters, merge with `merge_and_unload()`, push to the Hub

In [ ]:
ADAPTER_DIR, MERGED_DIR = f"{OUT}/adapter", f"{OUT}/merged"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

from task2_genai.finetune import free_gpu, merge_and_save
del model, trainer
free_gpu()
merge_and_save(cfg, ADAPTER_DIR, MERGED_DIR, tokenizer)
print(os.listdir(MERGED_DIR))

In [ ]:
from task2_genai.finetune import push_to_hub
CARD = f"""---
base_model: {cfg.base_model}
library_name: transformers
tags: [qlora, finance, risk-extraction]
---
# Qwen2.5-3B risk-factor extractor

QLoRA (4-bit NF4, r={cfg.lora_r}, alpha={cfg.lora_alpha}) fine-tune of {cfg.base_model} that extracts material risk factors from financial-disclosure passages into JSON (12-category taxonomy, severity, affected metric, verbatim evidence quote). Adapters merged with merge_and_unload(). Trained on synthetic passages generated by Llama-family teacher models. Not investment advice.\n"""
url = push_to_hub(MERGED_DIR, HF_REPO_ID, HF_TOKEN, CARD)
print("Model:", url)

In [ ]:
from task2_genai.evaluation import generate, load_for_inference, prompt_messages
test = load_jsonl(DATA_DIR / "test.jsonl")
ft_model, ft_tok = load_for_inference(MERGED_DIR)
out = generate(ft_model, ft_tok, [prompt_messages(test[0])])[0]
print(prompt_messages(test[0])[1]["content"][:600])
print("\nMODEL OUTPUT:\n", out["text"])